<a href="https://colab.research.google.com/github/alvarezht/Prediccion-de-Win-Rate-en-Cartas-Magic-The-Gathering/blob/main/Magic_The_Gathering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PREPARACIONES

In [29]:
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import seaborn as sns
import os

AUTOTUNE = tf.data.AUTOTUNE

# DATASET PREPARATIONS

## Cargar dataset

In [30]:
raw_dataset = pd.read_csv("./dataset.csv")

## Inspeccionar

In [31]:
print("\n--- RESUMEN ESTADISTICO (.describe) ---")
print(raw_dataset.describe())

print("\n--- PRIMERAS 5 FILAS (.head) ---")
print(raw_dataset.head())

print("\n--- COLUMNAS (.columns) ---")
print(raw_dataset.columns.tolist())

print("\n--- SHAPE ---")
print(f"Filas: {raw_dataset.shape[0]}, Columnas: {raw_dataset.shape[1]}")


--- RESUMEN ESTADISTICO (.describe) ---
              # Seen        ALSA      # Picked         ATA           # GP  \
count     376.000000  376.000000    376.000000  375.000000     376.000000   
mean   121218.132979    4.536223  20456.154255    6.223147   77350.609043   
std    121504.152193    1.781483  17792.877739    2.957152   81683.375412   
min      1866.000000    1.230000    435.000000    1.220000      69.000000   
25%     20030.250000    3.117500   5104.000000    3.660000   18397.500000   
50%     87548.000000    4.550000  16935.000000    6.190000   50930.500000   
75%    153710.000000    5.840000  27703.750000    8.690000  113065.000000   
max    383024.000000    8.350000  68560.000000   12.060000  394270.000000   

               # OH          # GD          # GIH          # GNS  
count    376.000000    376.000000     376.000000     376.000000  
mean   13210.587766  18961.156915   32171.744681   44796.146277  
std    14152.920385  19696.313117   33786.970651   47476.640495  
m

## Seleccionar variables

In [32]:
independent_variables = ['# Seen', 'ALSA', '# Picked', 'ATA', '% GP']
dependent_variable = 'GIH WR'

print(f"\n[DATA] Variables independientes ({len(independent_variables)}): {independent_variables}")
print(f"[TARGET] Variable dependiente: {dependent_variable}")


[DATA] Variables independientes (5): ['# Seen', 'ALSA', '# Picked', 'ATA', '% GP']
[TARGET] Variable dependiente: GIH WR


## Limpiar valores nulos

In [33]:
new_dataset = raw_dataset.copy()
print(f"\n--- VALORES NULOS ---")
print(new_dataset.isna().sum())



--- VALORES NULOS ---
Name         0
Color       46
Rarity       0
# Seen       0
ALSA         0
# Picked     0
ATA          1
# GP         0
% GP         0
GP WR        8
# OH         0
OH WR       31
# GD         0
GD WR       23
# GIH        0
GIH WR      16
# GNS        0
GNS WR      12
IWD         16
dtype: int64


## Limpiar columnas con formato especial

In [34]:
# Varias columnas tienen valores como "40.4%", "3.6%", "-10.2pp", ""
def clean_pct(value):
    if isinstance(value, str):
        value = value.replace('%', '').replace('pp', '').strip()
        if value == '':
            return np.nan
    try:
        return float(value)
    except (ValueError, TypeError):
        return np.nan

# Aplicar limpieza a TODAS las columnas que usa el modelo
new_dataset['GIH WR'] = new_dataset['GIH WR'].apply(clean_pct)
for col in independent_variables:
    new_dataset[col] = new_dataset[col].apply(clean_pct)

new_dataset = new_dataset.dropna(subset=['GIH WR'] + independent_variables)
print(f"Filas despues de limpiar: {new_dataset.shape[0]}")

Filas despues de limpiar: 359


## Split del dataset

In [35]:
train, test = train_test_split(new_dataset, test_size=0.2, random_state=42)

train_set = train[independent_variables].astype(np.float32)
train_target = train[dependent_variable].astype(np.float32)

test_set = test[independent_variables].astype(np.float32)
test_target = test[dependent_variable].astype(np.float32)

print(f"\n--- SPLIT ---")
print(f"Training: {train_set.shape[0]} filas")
print(f"Test:     {test_set.shape[0]} filas")


--- SPLIT ---
Training: 287 filas
Test:     72 filas


## Visualizacion de datos

In [36]:
print("\n--- VISUALIZACION (pairplot) ---")
sns.pairplot(pd.concat([train_set, train_target.rename('GIH_WR')], axis=1), diag_kind="kde")
plt.savefig("pairplot.png", dpi=100)
plt.close()
print("Guardado: pipeline/pairplot.png")


--- VISUALIZACION (pairplot) ---
Guardado: pipeline/pairplot.png


# NORMALIZACION

In [37]:
train_mean = train_set.mean()
train_std = train_set.std()

def normalize(data, mean, std):
    return (data - mean) / (std + 1e-8)

normed_train = normalize(train_set, train_mean, train_std)
normed_test = normalize(test_set, train_mean, train_std)

# MODELO

In [38]:
# Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True
)

# Arquitectura
model = tf.keras.models.Sequential([
    tf.keras.layers.InputLayer(input_shape=(len(independent_variables),)),
    tf.keras.layers.Dense(units=64, activation='relu'),
    tf.keras.layers.Dropout(rate=0.15),
    tf.keras.layers.Dense(units=32, activation='relu'),
    tf.keras.layers.Dense(units=16, activation='relu'),
    tf.keras.layers.Dense(units=1, activation='linear')
])

# Compilar
model.compile(
    loss='mse',
    optimizer='adam',
    metrics=[tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,009 (11.75 KB)

 Trainable params: 3,009 (11.75 KB)

 Non-trainable params: 0 (0.00 B)

# ENTRENAMIENTO

In [39]:
history = model.fit(
    normed_train,
    train_target,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - loss: 2987.6519 - rmse: 54.6594 - val_loss: 2978.0938 - val_rmse: 54.5719
Epoch 2/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2968.3218 - rmse: 54.4823 - val_loss: 2963.3286 - val_rmse: 54.4365
Epoch 3/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2951.0366 - rmse: 54.3234 - val_loss: 2945.3628 - val_rmse: 54.2712
Epoch 4/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2930.4705 - rmse: 54.1338 - val_loss: 2920.1360 - val_rmse: 54.0383
Epoch 5/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 2902.9946 - rmse: 53.8794 - val_loss: 2883.4116 - val_rmse: 53.6974
Epoch 6/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2860.6626 - rmse: 53.4852 - val_loss: 2832.4829 - val_rmse: 53.2211
Epoch 7/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2803.1577 - rmse: 52.9449 - val_loss: 2759.1157 - val_rmse: 52.5273
Epoch 8/200
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2722.2727 - rmse: 52.1754 - val_loss: 2655.5178 - val_rmse: 

# EVALUACION

In [40]:

loss, rmse = model.evaluate(normed_test, test_target, verbose=0)
print(f"Test Loss (MSE): {loss:.4f}")
print(f"Test RMSE:       {rmse:.4f}")

Test Loss (MSE): 6.5923
Test RMSE:       2.5675


# RESULTADO FINAL

In [41]:
print("\n" + "=" * 60)
print("RESULTADO FINAL")
print("=" * 60)
print(f"Modelo entrenado en {train_set.shape[0]} cartas")
print(f"Evaluado en {test_set.shape[0]} cartas")
print(f"RMSE en test: {rmse:.2f} pp (puntos porcentuales)")
print(f"El modelo predice GIH WR con un error promedio de +/-{rmse:.2f} pp")
print("=" * 60)


RESULTADO FINAL
Modelo entrenado en 287 cartas
Evaluado en 72 cartas
RMSE en test: 2.57 pp (puntos porcentuales)
El modelo predice GIH WR con un error promedio de +/-2.57 pp
